# Aula 9 — Correlação não é causalidade — Oficina 2 e AV2

**Habilidade:** Interpretar a relação entre variáveis por meio de gráficos de dispersão, covariância e correlação, distinguindo associação de causalidade.

**Objetos:** Correlação e causalidade. Variáveis de confusão.

Compare uma correlação forte, uma fraca, uma que “anda junto” só por tendência temporal e um caso de variável de confusão simulado.

> Base **simulada** (didática). Se o download falhar, envie `rio_negro_cotas.csv` pelo painel de arquivos do Colab.

In [ ]:
# Passo 0 — baixar o CSV do projeto (executar uma vez)
import os, urllib.request
if not os.path.exists("rio_negro_cotas.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/priscilagon/matematica-estatistica-ia/main/dados/rio_negro_cotas.csv", "rio_negro_cotas.csv")
print("CSV pronto:", os.path.exists("rio_negro_cotas.csv"))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("rio_negro_cotas.csv", parse_dates=["data"])

# Aula 9 — Correlação não é causalidade (Oficina 2)
d = df[df["temperatura_c"] < 35].copy()

# 1) Uma correlação FORTE nos dados do projeto
print("cota x temperatura: r = %.3f" % d["cota_metros"].corr(d["temperatura_c"]))
print("  -> a temperatura 'causa' a cota? Improvável: a ESTAÇÃO do ano move as duas.")

# 2) Uma correlação FRACA nos dados do projeto
print("\ncota x chuva (mesmo dia): r = %.3f" % d["cota_metros"].corr(d["chuva_mm"]))
d["chuva_30d"] = d["chuva_mm"].rolling(30).sum()
print("cota x chuva acumulada dos 30 dias anteriores: r = %.3f" % d["cota_metros"].corr(d["chuva_30d"]))
print("  -> r perto de zero mede só a relação LINEAR; não prova que 'não há relação'.")

# 3) Séries que apenas "andam juntas" com o tempo
d["dia_do_ano"] = d["data"].dt.dayofyear
subida = d[d["dia_do_ano"] <= 180]
print("\ncota x dia do ano (da enchente até o pico): r = %.3f" %
      subida["cota_metros"].corr(subida["dia_do_ano"]))

# 4) Variável de confusão — experimento SIMULADO (dados fictícios)
rng = np.random.default_rng(9)
calor = rng.normal(30, 4, 200)                           # causa comum (oculta)
sorvete = 20 + 3 * calor + rng.normal(0, 8, 200)         # vendas de sorvete
afogamentos = 0.5 * calor + rng.normal(0, 3, 200)        # acidentes na água
print("\nsorvete x afogamentos: r = %.3f" % np.corrcoef(sorvete, afogamentos)[0, 1])

def residuo(y, x):                                       # tira o efeito de x sobre y
    a, b = np.polyfit(x, y, 1)
    return y - (a * x + b)

r_ctrl = np.corrcoef(residuo(sorvete, calor), residuo(afogamentos, calor))[0, 1]
print("controlando o calor:     r = %.3f" % r_ctrl)

## O que observar

- `cota × dia do ano` tem r = 0,943 apenas na fase de enchente: ambos crescem com o tempo.
- `sorvete × afogamentos` cai de 0,535 para 0,008 quando se controla o calor (dados **fictícios**).
- r próximo de zero não prova ausência de relação (pode ser não linear ou defasada).

## Atividades de fixação

1. Um estudo encontrou correlação r = 0,85 entre o número de jogadores online e as quedas de energia em uma cidade. Liste duas explicações não causais possíveis.
2. Explique, com suas palavras, o que é variável de confusão e dê um exemplo do Rio Negro.
3. Reescreva a frase “o aumento da temperatura reduz a cota do rio” de modo cientificamente mais cuidadoso.
4. Cite três condições que reforçam um argumento de causalidade.


## Minhas respostas

(Use células de texto e de código abaixo.)